In [2]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

df = pd.read_csv("IBM-HR-Employee-Attrition.csv")

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [ ]:
#PHASE 1: Looking at what combination of overtime and commute affect attrition for Tesla's Gigafactory

In [4]:
#looking at employees who are working heavy overtime and have to drive a long distance to get to work

df_overtime = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE OverTime = 'Yes'
""", conn)

df_overtime.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
2,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
3,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
4,29,No,Travel_Rarely,153,Research & Development,15,2,Life Sciences,1,15,...,4,80,0,10,3,3,9,5,0,8


In [6]:
pd.read_sql_query("""
SELECT 
    Attrition,
    OverTime,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY Attrition, OverTime
ORDER BY Attrition, OverTime
""", conn)


,Attrition,OverTime,EmployeeCount
0,No,No,944
1,No,Yes,289
2,Yes,No,110
3,Yes,Yes,127


In [8]:
matrix_counts = pd.read_sql_query("""
SELECT 
    OverTime,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM employee_attrition
GROUP BY OverTime
ORDER BY OverTime
""", conn)

matrix_percentage = matrix_counts.set_index("OverTime")
matrix_percentage = matrix_percentage.div(matrix_percentage.sum(axis=1), axis=0) * 100
matrix_percentage = matrix_percentage.round(1)

print(matrix_percentage.astype(str) + '%')


             No    Yes
OverTime              
No        89.6%  10.4%
Yes       69.5%  30.5%


In [10]:
distance_counts = pd.read_sql_query("""
SELECT
    Distance_Bucket,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM (
    SELECT *,
        CASE
            WHEN DistanceFromHome BETWEEN 0 AND 5 THEN '0-5 miles'
            WHEN DistanceFromHome > 5 AND DistanceFromHome <= 10 THEN '5-10 miles'
            WHEN DistanceFromHome > 10 AND DistanceFromHome <= 15 THEN '10-15 miles'
            WHEN DistanceFromHome > 15 AND DistanceFromHome <= 20 THEN '15-20 miles'
            WHEN DistanceFromHome > 20 AND DistanceFromHome <= 25 THEN '20-25 miles'
            WHEN DistanceFromHome > 25 AND DistanceFromHome <= 30 THEN '25-30 miles'
        END AS Distance_Bucket
    FROM employee_attrition
)
GROUP BY Distance_Bucket
ORDER BY
    CASE Distance_Bucket
        WHEN '0-5 miles' THEN 1
        WHEN '5-10 miles' THEN 2
        WHEN '10-15 miles' THEN 3
        WHEN '15-20 miles' THEN 4
        WHEN '20-25 miles' THEN 5
        WHEN '25-30 miles' THEN 6
    END
""", conn)

chart_matrix = distance_counts.set_index("Distance_Bucket")
print(chart_matrix)


                  No  Yes
Distance_Bucket          
0-5 miles        545   87
5-10 miles       337   57
10-15 miles       90   25
15-20 miles      102   23
20-25 miles       85   32
25-30 miles       74   13


In [12]:
percentage_matrix = chart_matrix.div(chart_matrix.sum(axis=1), axis=0) * 100
percentage_matrix = percentage_matrix.round(1)

print(percentage_matrix.astype(str) + '%')


                    No    Yes
Distance_Bucket              
0-5 miles        86.2%  13.8%
5-10 miles       85.5%  14.5%
10-15 miles      78.3%  21.7%
15-20 miles      81.6%  18.4%
20-25 miles      72.6%  27.4%
25-30 miles      85.1%  14.9%


In [15]:
burnout_df = pd.read_sql_query("""
SELECT *,
    CASE
        WHEN OverTime = 'Yes' AND DistanceFromHome > 10 THEN 'High Risk'
        ELSE 'Standard Risk'
    END AS Gigafactory_Burnout_Risk
FROM employee_attrition
""", conn)

df = burnout_df.copy()


In [17]:
final_burnout_matrix = pd.crosstab(
    df['Gigafactory_Burnout_Risk'],
    df['Attrition'],
    normalize='index'
) * 100

print(final_burnout_matrix.round(1).astype(str) + '%')


Attrition                    No    Yes
Gigafactory_Burnout_Risk              
High Risk                 59.9%  40.1%
Standard Risk             86.3%  13.7%


In [19]:
#Statistical Testing for part 1

#Chi-Square on Burnout Risk Flag
contingency_table = pd.crosstab(df["Gigafactory_Burnout_Risk"], df["Attrition"])

chi2, p_val, dof, expected = chi2_contingency(contingency_table)
print("--- CHI-SQUARE TEST RESULTS ---")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"p-value: {p_val:.4e}")
print(f"Degrees of Freedom: {dof}\n")


--- CHI-SQUARE TEST RESULTS ---
Chi-Square Statistic: 62.5328
p-value: 2.6204e-15
Degrees of Freedom: 1



In [21]:
#Logistic Regression

df["Attrition_Numeric"] = df["Attrition"].map({"Yes": 1, "No": 0})
df["OverTime_Numeric"] = df["OverTime"].map({"Yes": 1, "No": 0})

model = smf.logit(
    "Attrition_Numeric ~ OverTime_Numeric + DistanceFromHome",
    data=df
).fit()

print("\n--- LOGISTIC REGRESSION SUMMARY ---")
print(model.summary())


Optimization terminated successfully.
         Current function value: 0.411346
         Iterations 6

--- LOGISTIC REGRESSION SUMMARY ---
                           Logit Regression Results                           
Dep. Variable:      Attrition_Numeric   No. Observations:                 1470
Model:                          Logit   Df Residuals:                     1467
Method:                           MLE   Df Model:                            2
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.06871
Time:                        03:26:25   Log-Likelihood:                -604.68
converged:                       True   LL-Null:                       -649.29
Covariance Type:            nonrobust   LLR p-value:                 4.216e-20
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -2.3886      0.135    -17.637      0.00

In [23]:
#PART 2: Does giving employees a higher stake in the company through stock offers actually acts as a potential shield  against burnout? 
#...Does stock ownership keep employees loyal and motivated even when overtime hours are needed?

equity_matrix = pd.read_sql_query("""
SELECT
    StockOptionLevel,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM employee_attrition
WHERE OverTime = 'Yes'
GROUP BY StockOptionLevel
ORDER BY StockOptionLevel
""", conn)

equity_matrix = equity_matrix.set_index("StockOptionLevel")
equity_matrix = equity_matrix.div(equity_matrix.sum(axis=1), axis=0) * 100

print(equity_matrix.round(1).astype(str) + '%')


                     No    Yes
StockOptionLevel              
0                 54.9%  45.1%
1                 82.6%  17.4%
2                 84.2%  15.8%
3                 65.5%  34.5%


In [25]:
#Isolating the employees who do not work overime:

df_no_overtime = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE OverTime = 'No'
""", conn)

no_overtime_matrix = pd.read_sql_query("""
SELECT
    StockOptionLevel,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM employee_attrition
WHERE OverTime = 'No'
GROUP BY StockOptionLevel
ORDER BY StockOptionLevel
""", conn)

no_overtime_matrix = no_overtime_matrix.set_index("StockOptionLevel")
no_overtime_matrix = no_overtime_matrix.div(no_overtime_matrix.sum(axis=1), axis=0) * 100

print("=== Regular Hours Workers: Attrition Rate by Stock Option Level ===")
print(no_overtime_matrix.round(1).astype(str) + '%')


=== Regular Hours Workers: Attrition Rate by Stock Option Level ===
                     No    Yes
StockOptionLevel              
0                 84.0%  16.0%
1                 93.7%   6.3%
2                 95.0%   5.0%
3                 91.1%   8.9%


In [27]:
comparison_table = pd.concat(
    [equity_matrix.round(1), no_overtime_matrix.round(1)],
    axis=1,
    keys=['Overtime Workers', 'Regular Workers']
)

print(comparison_table.astype(str) + '%')


                 Overtime Workers        Regular Workers       
                               No    Yes              No    Yes
StockOptionLevel                                               
0                           54.9%  45.1%           84.0%  16.0%
1                           82.6%  17.4%           93.7%   6.3%
2                           84.2%  15.8%           95.0%   5.0%
3                           65.5%  34.5%           91.1%   8.9%


In [29]:
# Chi-Square Test for people working overtime:
ct_overtime = pd.crosstab(df_overtime["StockOptionLevel"], df_overtime["Attrition"])
chi2_ot, p_ot, _, _ = chi2_contingency(ct_overtime)
print(f"Overtime Group Chi-Square p-value: {p_ot:.4e}")

# Chi-Square Test for Regular Workers
ct_regular = pd.crosstab(
    df_no_overtime["StockOptionLevel"],
    df_no_overtime["Attrition"]
)
chi2_reg, p_reg, _, _ = chi2_contingency(ct_regular)
print(f"Regular Group Chi-Square p-value: {p_reg:.4e}")


Overtime Group Chi-Square p-value: 8.0250e-08
Regular Group Chi-Square p-value: 6.2781e-06


In [31]:
# Logistic Regression

df["Attrition_Numeric"] = df["Attrition"].map({"Yes": 1, "No": 0})
df["OverTime_Numeric"] = df["OverTime"].map({"Yes": 1, "No": 0})

model = smf.logit(
    "Attrition_Numeric ~ OverTime_Numeric * C(StockOptionLevel)",
    data=df
).fit()

print(model.summary())


Optimization terminated successfully.
         Current function value: 0.392354
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:      Attrition_Numeric   No. Observations:                 1470
Model:                          Logit   Df Residuals:                     1462
Method:                           MLE   Df Model:                            7
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                  0.1117
Time:                        03:27:14   Log-Likelihood:                -576.76
converged:                       True   LL-Null:                       -649.29
Covariance Type:            nonrobust   LLR p-value:                 4.418e-28
                                                coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                    -

In [33]:
#PHASE 3: Do premium employee benefits offset intense workspaces in Tesla factories? How does employee work environments balance lower salaries or 
#..stressful travel requirements?

travel_env_counts = pd.read_sql_query("""
SELECT
    EnvironmentSatisfaction,
    BusinessTravel,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM employee_attrition
GROUP BY EnvironmentSatisfaction, BusinessTravel
ORDER BY EnvironmentSatisfaction, BusinessTravel
""", conn)

travel_env_counts["Total"] = travel_env_counts["No"] + travel_env_counts["Yes"]
travel_env_counts["No"] = (travel_env_counts["No"] / travel_env_counts["Total"]) * 100
travel_env_counts["Yes"] = (travel_env_counts["Yes"] / travel_env_counts["Total"]) * 100
travel_env_counts = travel_env_counts.drop(columns="Total")

travel_env_matrix = travel_env_counts.set_index(
    ["EnvironmentSatisfaction", "BusinessTravel"]
).unstack()

print("=== Attrition Rates: Environment Satisfaction vs. Business Travel ===")
print(travel_env_matrix.round(1).astype(str) + '%')


=== Attrition Rates: Environment Satisfaction vs. Business Travel ===
                                No                                        Yes  \
BusinessTravel          Non-Travel Travel_Frequently Travel_Rarely Non-Travel   
EnvironmentSatisfaction                                                         
1                            83.3%             66.1%         75.9%      16.7%   
2                            87.0%             74.5%         87.3%      13.0%   
3                            96.3%             74.7%         87.7%       3.7%   
4                            95.3%             82.1%         86.5%       4.7%   

                                                         
BusinessTravel          Travel_Frequently Travel_Rarely  
EnvironmentSatisfaction                                  
1                                   33.9%         24.1%  
2                                   25.5%         12.7%  
3                                   25.3%         12.3%  
4             

In [35]:
# Separating employees who have salaries below median income:

median_pay = df['MonthlyIncome'].median()

df_low_pay = pd.read_sql_query(f"""
SELECT *
FROM employee_attrition
WHERE MonthlyIncome < {median_pay}
""", conn)


In [37]:
true_env = pd.crosstab(
    df_low_pay['EnvironmentSatisfaction'],
    df_low_pay['Attrition'],
    normalize='index'
) * 100

true_job = pd.crosstab(
    df_low_pay['JobSatisfaction'],
    df_low_pay['Attrition'],
    normalize='index'
) * 100

print("=== True Low Earners Attrition: Physical Environment ===")
print(true_env.round(1))

print("\n=== True Low Earners Attrition: Job Satisfaction ===")
print(true_job.round(1))


=== True Low Earners Attrition: Physical Environment ===
Attrition                  No   Yes
EnvironmentSatisfaction            
1                        66.4  33.6
2                        83.0  17.0
3                        80.3  19.7
4                        80.5  19.5

=== True Low Earners Attrition: Job Satisfaction ===
Attrition          No   Yes
JobSatisfaction            
1                68.5  31.5
2                78.2  21.8
3                77.6  22.4
4                85.3  14.7


In [39]:
#Chi-Square Test: Environment Satisfaction vs Attrition for low-paid workers:
ct_low_pay_env = pd.crosstab(df_low_pay['EnvironmentSatisfaction'], df_low_pay['Attrition'])
chi2_env, p_env, _, _ = chi2_contingency(ct_low_pay_env)
print(f"Low Pay Environment Chi-Square p-value: {p_env:.4e}")

#Chi-Square Test: Job Satisfaction vs Attrition for Low Earners
ct_low_pay_job = pd.crosstab(df_low_pay['JobSatisfaction'], df_low_pay['Attrition'])
chi2_job, p_job, _, _ = chi2_contingency(ct_low_pay_job)
print(f"Low Pay Job Chi-Square p-value: {p_job:.4e}")


Low Pay Environment Chi-Square p-value: 2.2209e-03
Low Pay Job Chi-Square p-value: 2.4175e-03


In [41]:
# Logistic Regression

df['Attrition_Numeric'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df['Low_Pay_Numeric'] = (df['MonthlyIncome'] < df['MonthlyIncome'].median()).astype(int)

model = smf.logit(
    "Attrition_Numeric ~ C(BusinessTravel) + Low_Pay_Numeric + "
    "C(EnvironmentSatisfaction) + C(JobSatisfaction)",
    data=df
).fit()

print(model.summary())


Optimization terminated successfully.
         Current function value: 0.408189
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:      Attrition_Numeric   No. Observations:                 1470
Model:                          Logit   Df Residuals:                     1460
Method:                           MLE   Df Model:                            9
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.07586
Time:                        03:28:02   Log-Likelihood:                -600.04
converged:                       True   LL-Null:                       -649.29
Covariance Type:            nonrobust   LLR p-value:                 3.153e-17
                                             coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------
Intercept                                 -1.8985   

In [43]:
#PHASE 5: Does being underpaid compared to immediate peers at the same job level drive employees to quit,and does it impact high-performers more than
#...standard performers

tesla_titles = {
    1: 'Associate Analyst / Engineer',
    2: 'Analyst / Engineer',
    3: 'Senior Analyst / Senior Engineer',
    4: 'Staff Analyst / Staff Engineer',
    5: 'Senior Staff / Principal Track'
}

df['Job_Title'] = df['JobLevel'].map(tesla_titles)

df['Tier_Median_Income'] = df.groupby('JobLevel')['MonthlyIncome'].transform('median')

df['Tier_Pay_Position'] = np.where(
    df['MonthlyIncome'] < df['Tier_Median_Income'],
    'Below Tier Median',
    'At/Above Tier Median'
)


In [45]:
financial_matrix = pd.crosstab(
    index=[df['Tier_Pay_Position'], df['PerformanceRating']],
    columns=df['Attrition'],
    normalize='index'
) * 100

print("=== Financial Attrition: Tier Pay Position vs. Performance ===")
print(financial_matrix.round(1).astype(str) + '%')


=== Financial Attrition: Tier Pay Position vs. Performance ===
Attrition                                  No    Yes
Tier_Pay_Position    PerformanceRating              
At/Above Tier Median 3                  85.6%  14.4%
                     4                  89.1%  10.9%
Below Tier Median    3                  82.2%  17.8%
                     4                  78.4%  21.6%


In [46]:
contingency_table = pd.crosstab(
    index=[df["Tier_Pay_Position"], df["PerformanceRating"]],
    columns=df["Attrition"],
)

chi2, p_val, dof, expected = chi2_contingency(contingency_table)
print(f"Financial Attrition Matrix p-value: {p_val:.4e}")

#THIS ANALYSIS IS NOT STATISTICALLY RELEVANT


Financial Attrition Matrix p-value: 5.7130e-02
